In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parents[1]
SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

from models.logistic_regression import LogisticRegressionModel
from models.xgboost_model import XGBoostModel
from evaluation.shap_explainer import SHAPExplainer
from models.lightgbm_model import LightGBMModel
import shap

In [ ]:
FEATURE_PATH = PROJECT_ROOT / "data" / "processed" / "model_features"

X_train = pd.read_pickle(FEATURE_PATH / "X_train.pkl")
X_validation = pd.read_pickle(FEATURE_PATH / "X_validation.pkl")
y_train = pd.read_pickle(FEATURE_PATH / "y_train.pkl")
y_validation = pd.read_pickle(FEATURE_PATH / "y_validation.pkl")

print("X_train:", X_train.shape)
print("X_validation:", X_validation.shape)
print("y_train:", y_train.shape)
print("y_validation:", y_validation.shape)

In [ ]:
print("Train fraud rate:", y_train.mean())
print("Validation fraud rate:", y_validation.mean())

# Class weights

In [ ]:
fraud_count = (y_train == 1).sum()
legitimate_count = (y_train == 0).sum()
scale_ratio = legitimate_count / fraud_count

print("Legitimate transactions:", legitimate_count)
print("Fraud transactions:", fraud_count)
print("Fraud rate:", y_train.mean())
print("Class ratio:", scale_ratio)

# Instantiate the baseline model

In [ ]:
model = LogisticRegressionModel()
model.train(X_train, y_train)
print("Logistic Regression training completed.")

# Fraud probabilities

In [ ]:
y_validation_proba = model.predict_proba(X_validation)

print("Predictions generated:", len(y_validation_proba))
print("Minimum probability:", y_validation_proba.min())
print("Maximum probability:", y_validation_proba.max())

In [ ]:
pr_auc = average_precision_score(y_validation, y_validation_proba)
roc_auc = roc_auc_score(y_validation, y_validation_proba)

print("PR-AUC:", pr_auc)
print("ROC-AUC:", roc_auc)

# Threshold evaluation

In [ ]:
threshold = 0.5
y_validation_pred = (y_validation_proba >= threshold).astype(int)

print("Precision:", precision_score(y_validation, y_validation_pred))
print("Recall:", recall_score(y_validation, y_validation_pred))
print("F1:", f1_score(y_validation, y_validation_pred))
print(confusion_matrix(y_validation, y_validation_pred))

In [ ]:
baseline_results = {
    "model": "Logistic Regression",
    "pr_auc": pr_auc,
    "roc_auc": roc_auc,
    "precision": precision_score(y_validation, y_validation_pred),
    "recall": recall_score(y_validation, y_validation_pred),
    "f1": f1_score(y_validation, y_validation_pred),
    "threshold": 0.5,
}
baseline_results

In [ ]:
thresholds = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
threshold_results = []

for t in thresholds:
    p = (y_validation_proba >= t).astype(int)
    threshold_results.append({
        "threshold": t,
        "precision": precision_score(y_validation, p),
        "recall": recall_score(y_validation, p),
        "f1": f1_score(y_validation, p),
    })

threshold_results = pd.DataFrame(threshold_results).sort_values("f1", ascending=False).reset_index(drop=True)
threshold_results

## Threshold Analysis Insight

- Increasing the decision threshold improves precision but reduces recall.
- At threshold 0.5, recall is high (70.8%) but precision is low (7.0%).
- Among the tested thresholds, 0.8 gives the highest F1 (~0.215), with
  precision ~15.9% and recall ~33.2%.
- Threshold selection represents a trade-off between catching more fraud
  and reducing false alerts.

# Xgboost

In [ ]:
xgb_model = XGBoostModel(scale_pos_weight=scale_ratio)
xgb_model.train(X_train, y_train, X_validation, y_validation)
print("XGBoost training completed.")

In [ ]:
y_validation_proba_xgb = xgb_model.predict_proba(X_validation)

print("Predictions generated:", len(y_validation_proba_xgb))
print("Minimum probability:", y_validation_proba_xgb.min())
print("Maximum probability:", y_validation_proba_xgb.max())

In [ ]:
pr_auc_xgb = average_precision_score(y_validation, y_validation_proba_xgb)
roc_auc_xgb = roc_auc_score(y_validation, y_validation_proba_xgb)

print("XGBoost PR-AUC:", pr_auc_xgb)
print("XGBoost ROC-AUC:", roc_auc_xgb)

In [ ]:
thresholds = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
xgb_threshold_results = []

for t in thresholds:
    p = (y_validation_proba_xgb >= t).astype(int)
    xgb_threshold_results.append({
        "threshold": t,
        "precision": precision_score(y_validation, p),
        "recall": recall_score(y_validation, p),
        "f1": f1_score(y_validation, p),
    })

xgb_threshold_results = pd.DataFrame(xgb_threshold_results).sort_values("f1", ascending=False).reset_index(drop=True)
xgb_threshold_results

In [ ]:
xgb_threshold = 0.7
y_validation_pred_xgb = (y_validation_proba_xgb >= xgb_threshold).astype(int)

print(confusion_matrix(y_validation, y_validation_pred_xgb))
print("Precision:", precision_score(y_validation, y_validation_pred_xgb))
print("Recall:", recall_score(y_validation, y_validation_pred_xgb))
print("F1:", f1_score(y_validation, y_validation_pred_xgb))

In [ ]:
feature_importance = pd.Series(
    xgb_model.model.feature_importances_,
    index=X_train.columns
).sort_values(ascending=False)

feature_importance.head(15)

# XGBoost Feature Importance Insight
----------------------------------

- ProductCD_C is the dominant feature in the baseline XGBoost model
  (importance ≈ 0.443).
- DeviceType_MISSING, card6 categories and several missingness indicators
  are also among the strongest features.
- Transaction amount features contribute to the model but have much lower
  importance than ProductCD_C.
- Entity and relationship features do not appear in the top 15 by the
  built-in XGBoost importance measure.
- Feature importance indicates model usage, not causality or fraud direction.
- The contribution of feature groups should therefore be validated through
  controlled ablation experiments before removing any engineered features.

# Feature-group ablation.

In [ ]:
transaction_features = ["TransactionAmt", "TransactionDT", "log_transaction_amount"]
missingness_features = ["addr1_missing", "addr2_missing", "D7_missing", "D12_missing", "D13_missing", "D14_missing", "DeviceInfo_missing"]
temporal_features = ["time_since_previous_transaction", "has_previous_transaction"]
entity_features = ["card1_transaction_count", "card2_transaction_count", "addr1_transaction_count"]
relationship_features = ["card1_DeviceInfo_transaction_count", "card1_unique_DeviceInfo_count", "DeviceInfo_unique_card1_count"]
categorical_encoded_features = [
    c for c in X_train.columns
    if c.startswith(("ProductCD_", "card4_", "card6_", "DeviceType_"))
]

print("Transaction:", len(transaction_features))
print("Missingness:", len(missingness_features))
print("Temporal:", len(temporal_features))
print("Entity:", len(entity_features))
print("Relationship:", len(relationship_features))
print("Categorical:", len(categorical_encoded_features))

In [ ]:
grouped_features = (
    transaction_features + missingness_features + temporal_features +
    entity_features + relationship_features + categorical_encoded_features
)

extra_columns = [c for c in X_train.columns if c not in grouped_features]
duplicates = pd.Series(grouped_features)[pd.Series(grouped_features).duplicated()].tolist()

print("X_train columns:", len(X_train.columns))
print("Grouped features:", len(grouped_features))
print("Unaccounted columns:", extra_columns)
print("Duplicate grouped features:", duplicates)

In [ ]:
base_features = grouped_features

ablation_configs = {
    "full": base_features,
    "without_missingness": [f for f in base_features if f not in missingness_features],
    "without_temporal": [f for f in base_features if f not in temporal_features],
    "without_entity": [f for f in base_features if f not in entity_features],
    "without_relationship": [f for f in base_features if f not in relationship_features],
}

for name, features in ablation_configs.items():
    print(f"{name}: {len(features)} features")

In [ ]:
ablation_results = []

for name, features in ablation_configs.items():
    m = XGBoostModel(scale_pos_weight=scale_ratio)
    m.train(X_train[features], y_train, X_validation[features], y_validation)
    prob = m.predict_proba(X_validation[features])

    ablation_results.append({
        "model": name,
        "pr_auc": average_precision_score(y_validation, prob),
        "roc_auc": roc_auc_score(y_validation, prob),
    })

ablation_results = pd.DataFrame(ablation_results).sort_values("pr_auc", ascending=False).reset_index(drop=True)
ablation_results

# Feature Ablation Insight
------------------------

- Entity features provide a strong contribution. Removing them reduces
  PR-AUC from 0.2104 to 0.1831 and ROC-AUC from 0.8097 to 0.7797.
- Missingness features are also important. Removing them reduces PR-AUC
  to 0.1838.
- Relationship features provide a smaller but measurable contribution,
  with PR-AUC decreasing from 0.2104 to 0.2073 when removed.
- Temporal features show little improvement in the current feature set.
  Removing them slightly reduces PR-AUC but increases ROC-AUC.
- Therefore, entity and missingness features are currently the strongest
  engineered feature groups, while relationship features should be
  retained and temporal features should be reconsidered during further
  development.

# XGBoost Tuning

In [ ]:
depth_results = []

for depth in [3, 6, 8]:
    m = XGBoostModel(scale_pos_weight=scale_ratio, max_depth=depth)
    m.train(X_train, y_train, X_validation, y_validation)
    prob = m.predict_proba(X_validation)

    depth_results.append({
        "max_depth": depth,
        "pr_auc": average_precision_score(y_validation, prob),
        "roc_auc": roc_auc_score(y_validation, prob),
    })

pd.DataFrame(depth_results).sort_values("pr_auc", ascending=False).reset_index(drop=True)

In [ ]:
child_results = []

for weight in [1, 5, 10]:
    m = XGBoostModel(
        scale_pos_weight=scale_ratio,
        max_depth=8,
        min_child_weight=weight
    )
    m.train(X_train, y_train, X_validation, y_validation)
    prob = m.predict_proba(X_validation)

    child_results.append({
        "min_child_weight": weight,
        "pr_auc": average_precision_score(y_validation, prob),
        "roc_auc": roc_auc_score(y_validation, prob),
    })

pd.DataFrame(child_results).sort_values("pr_auc", ascending=False).reset_index(drop=True)

In [ ]:
learning_results = []
for lr, n in [(0.03, 500), (0.05, 300), (0.10, 200)]:
    m = XGBoostModel(
        scale_pos_weight=scale_ratio,
        n_estimators=n,
        max_depth=8,
        min_child_weight=5,
        learning_rate=lr
    )
    m.train(X_train, y_train, X_validation, y_validation)
    prob = m.predict_proba(X_validation)

    learning_results.append({
        "learning_rate": lr,
        "n_estimators": n,
        "pr_auc": average_precision_score(y_validation, prob),
        "roc_auc": roc_auc_score(y_validation, prob),
    })

pd.DataFrame(learning_results).sort_values("pr_auc", ascending=False).reset_index(drop=True)

# Tune subsample and colsample_bytree

In [ ]:
sampling_results = []

for subsample in [0.7, 0.8, 1.0]:
    for colsample in [0.7, 0.8, 1.0]:
        m = XGBoostModel(
            scale_pos_weight=scale_ratio,
            n_estimators=300,
            max_depth=8,
            min_child_weight=5,
            learning_rate=0.05,
            subsample=subsample,
            colsample_bytree=colsample
        )
        m.train(X_train, y_train, X_validation, y_validation)
        prob = m.predict_proba(X_validation)

        sampling_results.append({
            "subsample": subsample,
            "colsample_bytree": colsample,
            "pr_auc": average_precision_score(y_validation, prob),
            "roc_auc": roc_auc_score(y_validation, prob),
        })

pd.DataFrame(sampling_results).sort_values("pr_auc", ascending=False).reset_index(drop=True)

# REGUALRIZATION

In [ ]:
l2_results = []

for reg_lambda in [1, 5, 10]:
    m = XGBoostModel(
        scale_pos_weight=scale_ratio,
        n_estimators=300,
        max_depth=8,
        min_child_weight=5,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0,
        reg_lambda=reg_lambda
    )
    m.train(X_train, y_train, X_validation, y_validation)
    prob = m.predict_proba(X_validation)

    l2_results.append({
        "reg_lambda": reg_lambda,
        "pr_auc": average_precision_score(y_validation, prob),
        "roc_auc": roc_auc_score(y_validation, prob),
    })

pd.DataFrame(l2_results).sort_values("pr_auc", ascending=False).reset_index(drop=True)

In [ ]:
l1_results = []

for alpha in [0, 0.1, 0.5, 1]:
    m = XGBoostModel(
        scale_pos_weight=scale_ratio,
        n_estimators=300,
        max_depth=8,
        min_child_weight=5,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=alpha,
        reg_lambda=1
    )
    m.train(X_train, y_train, X_validation, y_validation)
    prob = m.predict_proba(X_validation)

    l1_results.append({
        "reg_alpha": alpha,
        "pr_auc": average_precision_score(y_validation, prob),
        "roc_auc": roc_auc_score(y_validation, prob),
    })

pd.DataFrame(l1_results).sort_values("pr_auc", ascending=False).reset_index(drop=True)

# Scale_pos_weight

In [ ]:
weight_results = []

for factor in [0.5, 1.0, 1.5, 2.0]:
    m = XGBoostModel(
        scale_pos_weight=scale_ratio * factor,
        n_estimators=300,
        max_depth=8,
        min_child_weight=5,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0,
        reg_lambda=1
    )
    m.train(X_train, y_train, X_validation, y_validation)
    prob = m.predict_proba(X_validation)

    weight_results.append({
        "weight_factor": factor,
        "scale_pos_weight": scale_ratio * factor,
        "pr_auc": average_precision_score(y_validation, prob),
        "roc_auc": roc_auc_score(y_validation, prob),
    })

pd.DataFrame(weight_results).sort_values("pr_auc", ascending=False).reset_index(drop=True)

# Final XG boost model

In [ ]:
final_xgb = XGBoostModel(
    scale_pos_weight=13.730737,
    n_estimators=300,
    max_depth=8,
    min_child_weight=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0,
    reg_lambda=1
)
final_xgb.train(X_train, y_train, X_validation, y_validation)
print("Final XGBoost baseline trained.")

In [ ]:
final_proba = final_xgb.predict_proba(X_validation)
final_pr_auc = average_precision_score(y_validation, final_proba)
final_roc_auc = roc_auc_score(y_validation, final_proba)

print("Predictions:", len(final_proba))
print("PR-AUC:", final_pr_auc)
print("ROC-AUC:", final_roc_auc)

In [ ]:
final_threshold_results = []

for t in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]:
    p = (final_proba >= t).astype(int)
    final_threshold_results.append({
        "threshold": t,
        "precision": precision_score(y_validation, p),
        "recall": recall_score(y_validation, p),
        "f1": f1_score(y_validation, p),
    })

final_threshold_results = pd.DataFrame(final_threshold_results).sort_values("f1", ascending=False).reset_index(drop=True)
final_threshold_results

## Tuned XGBoost — Threshold Analysis Insight

- The tuned XGBoost model achieved a PR-AUC of **0.2260** and ROC-AUC of **0.8200**.
- Among the tested thresholds, **0.5** produced the highest F1 score.
- At threshold 0.5:
  - Precision = **25.80%**
  - Recall = **35.58%**
  - F1 = **29.91%**
- Lowering the threshold increases recall but decreases precision.
- Increasing the threshold increases precision but decreases recall.
- For example, threshold 0.2 gives **70.57% recall** but only **10.41% precision**, while threshold 0.8 gives **46.11% precision** but only **10.78% recall**.
- Therefore, threshold selection represents a trade-off between catching more fraud and reducing false alerts.
- **0.5 is the best tested threshold by F1**, but it is not yet considered the final production threshold.
- The tuned XGBoost model performs substantially better than our Logistic Regression baseline and will be used as the current candidate model for further interpretation and validation.

# SHAP

SHAP tells us how each feature contributed to a model's prediction.

In [ ]:
sample = X_validation.sample(5000, random_state=42)
explainer = shap.TreeExplainer(final_xgb.model)
shap_values = explainer.shap_values(sample)
print("SHAP shape:", shap_values.shape)

In [ ]:
shap.summary_plot(shap_values, sample, max_display=15)

# SHAP Interpretation Insight
---------------------------

- TransactionAmt is the most influential feature in the SHAP summary,
  followed by ProductCD_C and TransactionDT.
- High transaction amounts generally push predictions toward higher
  fraud risk.
- ProductCD_C has a strong positive contribution for many transactions,
  confirming the signal observed during EDA.
- Entity features such as card1_transaction_count and
  card1_unique_DeviceInfo_count have meaningful model contributions,
  supporting the value of entity-based feature engineering.
- Several missingness indicators also influence predictions, confirming
  that missingness can contain useful information.
- SHAP shows that a feature's effect is not always one-directional;
  the same feature can increase or decrease fraud risk depending on
  the transaction context.
- SHAP provides directional and transaction-level explanations that
  cannot be obtained from XGBoost's built-in feature importance alone.

# Local SHAP

In [ ]:
high_risk_idx = np.argmax(final_proba)
high_risk_score = final_proba[high_risk_idx]

print("Validation index:", high_risk_idx)
print("Fraud probability:", high_risk_score)
print("Actual label:", y_validation.iloc[high_risk_idx])

transaction = X_validation.iloc[[high_risk_idx]]
local_shap = explainer.shap_values(transaction)
print("Transaction shape:", transaction.shape)
print("SHAP shape:", local_shap.shape)

In [ ]:
shap.waterfall_plot(
    shap.Explanation(
        values=local_shap[0],
        base_values=explainer.expected_value,
        data=transaction.iloc[0],
        feature_names=transaction.columns,
    )
)

# Local SHAP Interpretation
-------------------------

- The selected validation transaction was actually fraudulent and received
  a predicted fraud probability of approximately 98.42%.
- The SHAP waterfall starts from the model's baseline output (-0.661) and
  shows how individual feature values push the prediction toward or away
  from fraud.
- ProductCD_C is the strongest displayed positive contributor (+0.98).
- Several D-feature indicators also make substantial contributions,
  demonstrating that the specific missing/present pattern matters.
- card2_transaction_count and card1_unique_DeviceInfo_count contribute
  positively, showing that entity and relationship features can influence
  an individual fraud decision.
- The final SHAP output is 4.129 in the model's raw log-odds space,
  corresponding to a fraud probability of approximately 98.42%.
- SHAP explains model contribution, not causal relationships.

# SHAP EXPLAINER

In [ ]:
shap_explainer = SHAPExplainer(final_xgb.model)
explanation = shap_explainer.explain(transaction, top_n=10)
explanation

In [ ]:
fraud_reasons = explanation[explanation["shap_value"] > 0]
legitimate_reasons = explanation[explanation["shap_value"] < 0]

result = shap_explainer.explain_prediction(transaction, top_n=5)
print("Fraud probability:", result["fraud_probability"])
result["reasons"]

In [ ]:
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "extracted_data"

transactions = (
    pd.read_csv(DATA_PATH / "train_transaction.csv")
    .sort_values("TransactionDT")
    .reset_index(drop=True)
)

split_index = int(len(transactions) * 0.80)
validation_data = transactions.iloc[split_index:].copy()

train_identity = pd.read_csv(DATA_PATH / "train_identity.csv")
validation_data = validation_data.merge(
    train_identity[["TransactionID", "DeviceInfo", "DeviceType"]],
    on="TransactionID",
    how="left"
)

print("Validation rows:", len(validation_data))
print("X_validation rows:", len(X_validation))

In [ ]:
pred = (final_proba >= 0.5).astype(int)

tp_idx = np.where((y_validation.values == 1) & (pred == 1))[0][0]
tn_idx = np.where((y_validation.values == 0) & (pred == 0))[0][0]
fp_idx = np.where((y_validation.values == 0) & (pred == 1))[0][0]
fn_idx = np.where((y_validation.values == 1) & (pred == 0))[0][0]

print("TP:", tp_idx)
print("TN:", tn_idx)
print("FP:", fp_idx)
print("FN:", fn_idx)

In [ ]:
tp_result = shap_explainer.explain_prediction(X_validation.iloc[[tp_idx]], top_n=5)
print("Fraud probability:", tp_result["fraud_probability"])
tp_result["reasons"]

In [ ]:
tn_result = shap_explainer.explain_prediction(X_validation.iloc[[tn_idx]], top_n=5)
print("Fraud probability:", tn_result["fraud_probability"])
tn_result["reasons"]

In [ ]:
fp_result = shap_explainer.explain_prediction(X_validation.iloc[[fp_idx]], top_n=5)
print("Fraud probability:", fp_result["fraud_probability"])
fp_result["reasons"]

In [ ]:
fn_result = shap_explainer.explain_prediction(X_validation.iloc[[fn_idx]], top_n=5)
print("Validation index:", fn_idx)
print("Fraud probability:", fn_result["fraud_probability"])
print("Actual label:", y_validation.iloc[fn_idx])
fn_result["reasons"]

# False Negative SHAP Insight
---------------------------

- The selected transaction was actually fraudulent but received only a
  35.87% fraud probability, below the 0.5 threshold.
- ProductCD_C and the DeviceType state pushed the prediction toward fraud.
- D7_missing, TransactionAmt, and card1_transaction_count pushed the
  prediction toward legitimate and outweighed the stronger fraud signals.
- This shows that some fraudulent transactions can exhibit patterns that
  appear legitimate to the model.
- False-negative explanations can help identify missing or weak contextual
  signals, but a single transaction should not be used to justify changing
  the feature set.

# Aggregate error analysis.

In [ ]:
tp = ((y_validation.values == 1) & (pred == 1)).sum()
tn = ((y_validation.values == 0) & (pred == 0)).sum()
fp = ((y_validation.values == 0) & (pred == 1)).sum()
fn = ((y_validation.values == 1) & (pred == 0)).sum()

print("TP:", tp)
print("TN:", tn)
print("FP:", fp)
print("FN:", fn)

In [ ]:
precision = tp / (tp + fp)
recall = tp / (tp + fn)

print("Precision:", precision)
print("Recall:", recall)
print("False Positive Rate:", fp / (fp + tn))
print("False Negative Rate:", fn / (fn + tp))

In [ ]:
fp_data = validation_data.loc[(y_validation.values == 0) & (pred == 1)].copy()
fn_data = validation_data.loc[(y_validation.values == 1) & (pred == 0)].copy()

print("False Positives:", len(fp_data))
print("False Negatives:", len(fn_data))

In [ ]:
print("FALSE POSITIVE ProductCD:")
print(fp_data["ProductCD"].value_counts(normalize=True).round(4))

print("\nFALSE NEGATIVE ProductCD:")
print(fn_data["ProductCD"].value_counts(normalize=True).round(4))

In [ ]:
product_error = pd.DataFrame({
    "actual": validation_data["ProductCD"],
    "y_true": y_validation.values,
    "y_pred": pred
})

product_error["fp"] = ((product_error["y_true"] == 0) & (product_error["y_pred"] == 1)).astype(int)
product_error["fn"] = ((product_error["y_true"] == 1) & (product_error["y_pred"] == 0)).astype(int)

product_error = product_error.groupby("actual").agg(
    transactions=("y_true", "size"),
    fraud_count=("y_true", "sum"),
    fp=("fp", "sum"),
    fn=("fn", "sum"),
)

product_error["fp_rate"] = product_error["fp"] / (product_error["transactions"] - product_error["fraud_count"])
product_error["fn_rate"] = product_error["fn"] / product_error["fraud_count"]

product_error.sort_values("fn_rate", ascending=False)

# ProductCD Error Analysis Insight
--------------------------------

- ProductCD=W has a very high false-negative rate (87.13%), indicating
  that the current model misses most fraud transactions in this category.
  Its false-positive rate is very low (0.80%), showing that the model is
  highly conservative for W.
- ProductCD=C has the highest false-positive rate (27.32%), indicating
  that the model frequently flags legitimate C transactions as fraud.
  Its false-negative rate is lower (37.11%) than W.
- This suggests category-dependent model behavior: the model is relatively
  aggressive for ProductCD=C and conservative for ProductCD=W.
- These results are hypotheses for further investigation, not evidence
  that ProductCD itself causes fraud or that category-specific thresholds
  should be deployed.

In [ ]:
print("FALSE POSITIVE amount:")
print(fp_data["TransactionAmt"].describe())

print("\nFALSE NEGATIVE amount:")
print(fn_data["TransactionAmt"].describe())

In [ ]:
error_amount = pd.DataFrame({
    "ProductCD": validation_data["ProductCD"],
    "TransactionAmt": validation_data["TransactionAmt"],
    "y_true": y_validation.values,
    "y_pred": pred
})

error_amount["error_type"] = np.select(
    [
        (error_amount["y_true"] == 0) & (error_amount["y_pred"] == 1),
        (error_amount["y_true"] == 1) & (error_amount["y_pred"] == 0),
    ],
    ["FP", "FN"],
    default="Other",
)

error_amount = error_amount[error_amount["error_type"].isin(["FP", "FN"])]

error_amount.groupby(["ProductCD", "error_type"])["TransactionAmt"].agg(
    count="count",
    median="median",
    mean="mean",
    q75=lambda x: x.quantile(0.75),
).round(2)

# ProductCD + Transaction Amount Error Insight
---------------------------------------------

- ProductCD=W has a large false-negative population, and its missed fraud
  transactions have substantially lower transaction amounts than its false
  positives (median 107.95 vs 311.94).
- This suggests that lower-to-moderate amount W fraud transactions may be
  harder for the current model to distinguish from legitimate transactions.
- ProductCD=C has a very large false-positive population, but the median
  transaction amounts of C false positives and false negatives are similar
  (35.35 vs 31.23).
- Therefore, transaction amount alone does not explain the high C false
  positive rate.
- These patterns are hypotheses for further investigation and do not imply
  that transaction amount or ProductCD causally produces the model errors.

In [ ]:
missing_columns = ["addr1", "addr2", "D7", "D12", "D13", "D14", "DeviceInfo"]

fp_mask = (y_validation.values == 0) & (pred == 1)
fn_mask = (y_validation.values == 1) & (pred == 0)

fp_missing = validation_data.loc[fp_mask, missing_columns].isna().mean()
fn_missing = validation_data.loc[fn_mask, missing_columns].isna().mean()

pd.DataFrame({
    "FP_missing_rate": fp_missing,
    "FN_missing_rate": fn_missing,
}).round(4)

# Missingness Error Analysis Insight
----------------------------------

- False negatives show very high missingness in D7, D12, D13 and D14,
  with rates above 77% and D7 reaching 83.23%.
- This suggests that many missed fraud transactions share substantial
  D-field missingness patterns.
- False positives show much higher addr1/addr2 missingness (68.78%) than
  false negatives (22.77%).
- DeviceInfo is missing in both error groups, but more frequently among
  false negatives (66.81% vs 54.86%).
- These results support the importance of explicit missingness features,
  while also showing that different missingness patterns are associated
  with different model errors.
- These are error-analysis associations, not causal conclusions.

In [ ]:
entity_features = [
    "card1_transaction_count",
    "card2_transaction_count",
    "addr1_transaction_count",
    "card1_DeviceInfo_transaction_count",
    "card1_unique_DeviceInfo_count",
    "DeviceInfo_unique_card1_count",
]

entity_error = pd.DataFrame({
    "FP": X_validation.loc[fp_mask, entity_features].median(),
    "FN": X_validation.loc[fn_mask, entity_features].median(),
})

entity_error

# Entity Error Analysis Insight
-----------------------------

- False negatives have higher median card1 and card2 transaction counts
  than false positives, with card2 showing the largest difference
  (9019 vs 3912).
- False positives have a much higher median addr1 transaction count
  (53761 vs 19826), suggesting that highly common address entities are
  more represented among incorrectly flagged legitimate transactions.
- card1_unique_DeviceInfo_count is also higher for false positives
  (65 vs 17), indicating greater device diversity in the FP group.
- The relationship-frequency features have zero medians in both error
  groups, so their usefulness cannot be determined from median values alone.
- These patterns show that entity behavior differs between FP and FN
  cases, but they represent associations rather than causal explanations.

In [ ]:
error_proba = pd.DataFrame({
    "error_type": np.select([fp_mask, fn_mask], ["FP", "FN"], default="Other"),
    "probability": final_proba,
})

error_proba = error_proba[error_proba["error_type"].isin(["FP", "FN"])]
error_proba.groupby("error_type")["probability"].describe().round(4)

# Prediction Confidence Error Insight
-----------------------------------

- False negatives have a low probability distribution, with median fraud
  probability of 0.2173 and 75th percentile of 0.3430.
- False positives have a much higher probability distribution, with median
  0.6252 and 25th percentile of 0.5585.
- Therefore, most missed fraud cases are not simply borderline cases just
  below the 0.5 threshold.
- Likewise, many false positives receive relatively high fraud scores
  rather than being marginal threshold cases.
- This suggests that the current model has genuine feature-separation
  limitations in addition to threshold-selection limitations.
- Threshold tuning can change the precision-recall trade-off, but it is
  unlikely to eliminate the underlying FP/FN errors by itself.

# LIGHTGBM

In [ ]:
lgbm_model = LightGBMModel(scale_pos_weight=13.730737)
print("LightGBM baseline initialized.")